# NVIDIA Relighting – Python Notebook

This notebook demonstrates how to use the NVIDIA Relighting NIM with Python.
The service applies AI-driven relighting to video — changing the illumination on faces
using HDR environment maps while preserving natural appearance.

## Overview

The service accepts an MP4 video and returns a relit MP4 video via bidirectional gRPC streaming:

- **HDR illumination** – built-in presets or custom `.hdr` files
- **Pan / vertical FOV** – rotate and zoom the HDR environment
- **Autorotate** – continuously spin the light during playback
- **Background source** – keep the original, replace with an image, or project the HDR map
- **Effect parameters** – foreground/background gain, blur, specular highlights
- **Inline image uploads** – HDRI and background images sent within the same gRPC stream

## Requirements

- **Input video** – MP4 with H.264 codec
- **Output** – MP4 with H.264 codec
- **Service** – a running Relighting NIM instance


## Installation

Install the Python dependencies from the relighting `requirements.txt`:


In [ ]:
!pip install -r ../requirements.txt

## Service Configuration

Configure the connection to your NVIDIA Relighting service.
To run the Relighting NIM follow the instructions in
[Getting Started](https://docs.nvidia.com/nim/maxine/relighting/latest/getting-started.html).


In [ ]:
import math
import os
import pathlib
import sys

SCRIPT_PATH = str(pathlib.Path().resolve())
sys.path.append(os.path.join(SCRIPT_PATH, "../scripts"))
sys.path.append(os.path.join(SCRIPT_PATH, "../interfaces"))
sys.path.append(os.path.join(SCRIPT_PATH, "../../"))

SERVICE_HOST = "localhost"
SERVICE_PORT = 8001
SERVICE_TARGET = f"{SERVICE_HOST}:{SERVICE_PORT}"

print(f"Service target: {SERVICE_TARGET}")

In [ ]:
import time
from collections.abc import Iterator
from pathlib import Path

import grpc
from nvidia.ai4m.relighting.v1 import relighting_pb2, relighting_pb2_grpc
from nvidia.ai4m.video.v1 import video_pb2

from utils import create_protobuf_any_value

DATA_CHUNK_SIZE = 64 * 1024  # 64 KiB per gRPC message

print("All libraries imported successfully!")

## Helper Functions

The relighting service uses bidirectional gRPC streaming:

1. **Config message** – relighting parameters (HDR, angles, effects, encoding)
2. **Image uploads** – optional inline HDRI and/or background image in 64 KiB chunks
3. **Video chunks** – the input MP4, streamed in 64 KiB chunks

The server responds with:

- `image_upload_ack` – confirms each image was received and decoded
- `progress` / `keep_alive` – processing status
- `video_data` – the relit output MP4 in chunks


In [ ]:
def build_video_encoding(
    bitrate_bps: int = 10_000_000,
    idr_interval: int = 8,
    lossless: bool = False,
    custom_params: dict | None = None,
) -> video_pb2.VideoEncoding:
    """Build a VideoEncoding proto from simple Python values."""
    enc = video_pb2.VideoEncoding()
    if lossless:
        enc.lossless = True
    elif custom_params:
        for k, v in custom_params.items():
            enc.custom_encoding.custom[k].CopyFrom(create_protobuf_any_value(v))
    else:
        mbps = max(1, round(bitrate_bps / 1_000_000)) if bitrate_bps > 0 else 0
        enc.lossy.CopyFrom(video_pb2.LossyEncoding(bitrate_mbps=mbps, idr_interval=idr_interval))
    return enc


def build_relight_config(
    hdri_preset_id: int = 0,
    hdri_image_bytes: bytes | None = None,
    pan_degrees: float = -90.0,
    vfov_degrees: float = 60.0,
    autorotate: bool = False,
    rotation_rate_degrees: float = 20.0,
    background_source: int = 0,
    background_image_type: int | None = None,
    background_color: int | None = None,
    foreground_gain: float = 1.0,
    background_gain: float = 1.0,
    blur_strength: float = 0.0,
    specular: float = 0.0,
    encoding: video_pb2.VideoEncoding | None = None,
) -> relighting_pb2.RelightConfig:
    """Build a RelightConfig proto from simple Python values."""
    rc = relighting_pb2.RelightConfig()

    if hdri_image_bytes is not None:
        rc.hdri_image_provided = True
    else:
        rc.hdri_preset_id = hdri_preset_id

    rc.angle_pan_radians = math.radians(pan_degrees)
    rc.angle_v_fov_radians = math.radians(vfov_degrees)
    rc.background_source = background_source
    if background_image_type is not None:
        rc.background_image_type = background_image_type
    if background_color is not None:
        rc.background_color = background_color
    rc.foreground_gain = foreground_gain
    rc.background_gain = background_gain
    rc.blur_strength = blur_strength
    rc.specular = specular
    rc.autorotate = autorotate
    rc.rotation_rate = math.radians(rotation_rate_degrees)

    if encoding is not None:
        rc.output_video_encoding.CopyFrom(encoding)
    else:
        rc.output_video_encoding.CopyFrom(build_video_encoding())

    return rc

In [ ]:
def generate_requests(
    video_path: str,
    config: relighting_pb2.RelightConfig,
    hdri_image_bytes: bytes | None = None,
    background_image_bytes: bytes | None = None,
) -> Iterator[relighting_pb2.RelightRequest]:
    """Stream RelightRequest messages: config, optional images, then video chunks."""
    # 1. Config
    yield relighting_pb2.RelightRequest(config=config)

    # 2. Inline images (before video)
    for img_bytes, img_type, label in [
        (hdri_image_bytes, relighting_pb2.IMAGE_TYPE_HDRI, "HDRI"),
        (background_image_bytes, relighting_pb2.IMAGE_TYPE_BACKGROUND, "Background"),
    ]:
        if img_bytes is not None:
            print(f"Sending {label} image ({len(img_bytes):,} bytes)...")
            for offset in range(0, len(img_bytes), DATA_CHUNK_SIZE):
                yield relighting_pb2.RelightRequest(
                    image_data=relighting_pb2.ImageData(
                        image_type=img_type,
                        data=img_bytes[offset : offset + DATA_CHUNK_SIZE],
                    )
                )

    # 3. Video
    print("Sending video data...")
    chunk_count = 0
    with open(video_path, "rb") as f:
        while chunk := f.read(DATA_CHUNK_SIZE):
            chunk_count += 1
            yield relighting_pb2.RelightRequest(video_data=chunk)
    print(f"Video chunks sent: {chunk_count}")

In [ ]:
def write_response_to_file(
    responses: Iterator[relighting_pb2.RelightResponse],
    output_path: str,
) -> None:
    """Consume the response stream and write the relit video to disk."""
    print(f"Writing output to: {output_path}")
    chunk_count = 0
    total_bytes = 0
    start = time.time()

    with open(output_path, "wb") as fd:
        for response in responses:
            if response.HasField("image_upload_ack"):
                ack = response.image_upload_ack
                label = "HDRI" if ack.image_type == 1 else "Background"
                print(f"{label} image accepted ({ack.size_bytes / 1024:.1f} KB)")

            elif response.HasField("progress"):
                p = response.progress
                if p.HasField("total_frames") and p.total_frames > 0:
                    pct = 100 * p.frames_processed // p.total_frames
                    print(
                        f"\r\033[KProcessing: {pct}% (frame {p.frames_processed}/{p.total_frames})",
                        end="",
                        flush=True,
                    )
                else:
                    print(f"\r\033[KProcessing: frame {p.frames_processed}", end="", flush=True)

            elif response.HasField("keep_alive"):
                print("\r\033[KProcessing...", end="", flush=True)

            elif response.HasField("video_data"):
                fd.write(response.video_data)
                chunk_count += 1
                total_bytes += len(response.video_data)

    elapsed = time.time() - start
    mb = total_bytes / (1024 * 1024)
    print(f"\nCompleted: {chunk_count} chunks | {mb:.2f} MB | {elapsed:.1f}s")

## Processing Function

A convenience wrapper that connects to the service, streams the request, and writes the output.


In [ ]:
def process_relighting(
    video_path: str,
    output_path: str,
    config: relighting_pb2.RelightConfig,
    hdri_image_path: str | None = None,
    background_image_path: str | None = None,
) -> bool:
    """Run relighting end-to-end: connect, stream, write output.

    Returns True on success, False on error.
    """
    hdri_bytes = None
    bg_bytes = None

    try:
        if hdri_image_path:
            hdri_bytes = Path(hdri_image_path).read_bytes()
        if background_image_path:
            bg_bytes = Path(background_image_path).read_bytes()

        print(f"Input:  {video_path}")
        print(f"Output: {output_path}")
        print(f"Server: {SERVICE_TARGET}")

        with grpc.insecure_channel(SERVICE_TARGET) as channel:
            stub = relighting_pb2_grpc.VideoRelightingServiceStub(channel)
            start = time.time()

            responses = stub.Relight(generate_requests(video_path, config, hdri_bytes, bg_bytes))
            write_response_to_file(responses, output_path)

            print(f"Relighting completed in {time.time() - start:.1f}s")
            print(f"Output saved: {output_path}")
            return True

    except FileNotFoundError as e:
        print(f"File not found: {e}")
        return False
    except grpc.RpcError as e:
        print(f"gRPC error: {e.code()} - {e.details()}")
        print(f"   Ensure the service is running at {SERVICE_TARGET}")
        return False
    except Exception as e:
        print(f"Error: {e}")
        return False

## Run Relighting – Basic Example

Relight a video using the default HDR preset and lossy encoding at 10 Mbps.


In [ ]:
video_filepath = "../assets/sample_video.mp4"  # Update with your video
output_filepath = "relighting_output.mp4"

config = build_relight_config()  # all defaults

success = process_relighting(video_filepath, output_filepath, config)

## HDR Presets

The service ships five built-in HDR environment maps:

| ID  | Name                      | Description                 |
| --- | ------------------------- | --------------------------- |
| 0   | Lounge                    | Lounge environment          |
| 1   | Cobblestone Street Night  | Cobblestone street at night |
| 2   | Glasshouse Interior       | Glasshouse interior         |
| 3   | Little Paris Eiffel Tower | Little Paris Eiffel Tower   |
| 4   | Wooden Studio             | Wooden studio environment   |


In [ ]:
preset_names = [
    "lounge",
    "cobblestone_street_night",
    "glasshouse_interior",
    "little_paris_eiffel_tower",
    "wooden_studio",
]

for preset_id, name in enumerate(preset_names):
    cfg = build_relight_config(hdri_preset_id=preset_id)
    out = f"output_preset_{name}.mp4"
    print(f"\n=== Preset {preset_id}: {name} ===")
    process_relighting(video_filepath, out, cfg)

## Pan Angle & Vertical FOV

- **Pan** rotates the HDR environment horizontally (–180° to 180°).
- **Vertical FOV** controls how much of the HDR map is visible (narrower = more zoomed in).


In [ ]:
pan_angles = [-180, -90, 0, 90, 180]

for pan in pan_angles:
    cfg = build_relight_config(pan_degrees=pan)
    out = f"output_pan_{pan}.mp4"
    print(f"\n=== Pan {pan}\u00b0 ===")
    process_relighting(video_filepath, out, cfg)

## Autorotate

When enabled, the pan angle continuously rotates during playback at the specified rate.


In [ ]:
cfg = build_relight_config(autorotate=True, rotation_rate_degrees=30.0)

print("=== Autorotate at 30\u00b0/s ===")
process_relighting(video_filepath, "output_autorotate.mp4", cfg)

## Background Source

| Value | Meaning                                       |
| ----- | --------------------------------------------- |
| 0     | Keep original video background (default)      |
| 1     | Replace with a custom image                   |
| 2     | Project the HDR environment map as background |

Combine any source with `blur_strength > 0` for a blurred variant.


In [ ]:
# Source video background (default)
cfg_src = build_relight_config(background_source=0)
print("=== Background: source video ===")
process_relighting(video_filepath, "output_bg_source.mp4", cfg_src)

# Source video background with blur
cfg_blur = build_relight_config(background_source=0, blur_strength=0.5)
print("\n=== Background: source video + blur ===")
process_relighting(video_filepath, "output_bg_source_blur.mp4", cfg_blur)

# HDR projection background
cfg_hdr = build_relight_config(background_source=2)
print("\n=== Background: HDR projection ===")
process_relighting(video_filepath, "output_bg_hdr.mp4", cfg_hdr)

### Custom Background Image

Upload a custom background image (PNG, JPG, or HDR) inline with `background_source=1`.


In [ ]:
background_image_path = None  # Set to your image path, e.g. "../assets/office.jpg"

if background_image_path and os.path.exists(background_image_path):
    cfg_img = build_relight_config(background_source=1)
    print("=== Background: custom image ===")
    process_relighting(
        video_filepath,
        "output_bg_image.mp4",
        cfg_img,
        background_image_path=background_image_path,
    )
else:
    print("Set background_image_path to a valid image to run this example.")

## Custom HDR File

Upload your own `.hdr` environment map instead of using a preset.


In [ ]:
custom_hdr_path = None  # Set to your .hdr file, e.g. "../assets/studio.hdr"

if custom_hdr_path and os.path.exists(custom_hdr_path):
    hdr_bytes = Path(custom_hdr_path).read_bytes()
    cfg_hdr = build_relight_config(hdri_image_bytes=hdr_bytes)
    print("=== Custom HDR ===")
    process_relighting(
        video_filepath,
        "output_custom_hdr.mp4",
        cfg_hdr,
        hdri_image_path=custom_hdr_path,
    )
else:
    print("Set custom_hdr_path to a valid .hdr file to run this example.")

## Effect Parameters

| Parameter         | Range   | Default | Description                           |
| ----------------- | ------- | ------- | ------------------------------------- |
| `foreground_gain` | 0.0–2.0 | 1.0     | Relighting strength on the foreground |
| `background_gain` | 0.0–2.0 | 1.0     | Relighting strength on the background |
| `blur_strength`   | 0.0–1.0 | 0.0     | Background blur amount                |
| `specular`        | 0.0–2.0 | 0.0     | Specular highlight intensity          |


In [ ]:
# Strong relighting with specular highlights
cfg = build_relight_config(foreground_gain=1.8, specular=1.0)
print("=== High gain + specular ===")
process_relighting(video_filepath, "output_high_gain.mp4", cfg)

# Subtle relighting
cfg = build_relight_config(foreground_gain=0.5, background_gain=0.5)
print("\n=== Subtle relighting ===")
process_relighting(video_filepath, "output_subtle.mp4", cfg)

## Encoding Options

### Lossy (default)

```python
build_video_encoding(bitrate_bps=10_000_000, idr_interval=8)
```

### Lossless

```python
build_video_encoding(lossless=True)
```

### Custom encoding parameters

```python
build_video_encoding(custom_params={"idrinterval": 20, "maxbitrate": 4000000})
```

**Note:** Custom parameters map directly to
[Gst-nvvideo4linux2 encoder properties](https://docs.nvidia.com/metropolis/deepstream/dev-guide/text/DS_plugin_gst-nvvideo4linux2.html).
Incorrect values can cause encoding failures.


In [ ]:
# High-bitrate lossy
enc_hq = build_video_encoding(bitrate_bps=40_000_000, idr_interval=4)
cfg_hq = build_relight_config(encoding=enc_hq)
print("=== High bitrate (40 Mbps) ===")
process_relighting(video_filepath, "output_hq.mp4", cfg_hq)

# Lossless
enc_ll = build_video_encoding(lossless=True)
cfg_ll = build_relight_config(encoding=enc_ll)
print("\n=== Lossless ===")
process_relighting(video_filepath, "output_lossless.mp4", cfg_ll)